In [ ]:
from mipt import *
import warnings
import numpy as np
import importlib
import csv
import gnme
importlib.reload(gnme)
from gnme import RingInflationSDP3Q
import matplotlib.pyplot as plt 

warnings.filterwarnings('ignore', category=UserWarning)

LEVEL = 2
STRATEGY = "feasibility"

sdp: RingInflationSDP3Q

n=8
d=2*n
ps = np.linspace(0.1, 0.5, 9)
reps = 25

if LEVEL == 2:
    sdp = RingInflationSDP3Q(domain="auto")
    with open("gnme_lvl2.csv", mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["p", "score"])
elif LEVEL == 3:
    # sdp = RingInflationSDP3QLevel3(
    #     domain="auto",          # "real", "complex", or "auto"
    #     strategy=STRATEGY  # usually more stable for SCS
    # )
    with open("gnme_lvl3.csv", mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["p", "score"])
else:
    raise ValueError("LEVEL must only be set to 2 or 3")

means, stderrs = [], []
for p in ps:
    print(f"Simulating p = {p:.2f}")
    scores = []
    for r in range(reps):
        rhos = get_mipt_rho_1d(n, d, p, subsyst=3, all_matrices=True, closed=True)
        gmn_scores = [gmn(rho, parties=3) for rho in rhos]
        idx = max(range(len(gmn_scores)), key=gmn_scores.__getitem__)
        rho = rhos[idx]

        score: float

        if LEVEL == 2:
            score = sdp.solve(
                rho,
                gpu=False,
                verbose=False,
                max_iters=1500,
                tol=1e-8,
                time_limit_secs=45,
            )
        elif LEVEL == 3:
            score = sdp.solve(
                rho,
                strategy=STRATEGY,
                gpu=True,
                max_iters=20_000,
                tol=2e-4,
                bisect_tol=2e-3,
                verbose=True,
            )

        new_row = [p, score]

        if LEVEL == 2:
            with open("gnme_lvl2.csv", mode="a", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(new_row)
        elif LEVEL == 3:
            with open("gnme_lvl3.csv", mode="a", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(new_row)

        scores.append(score)
        print(f"Realisations: {r}/{reps}", end='\r', flush=True)
        
    means.append(np.mean(scores))
    stderrs.append(np.std(scores)/np.sqrt(reps))
    print(f"\nInflation Score: {means[-1]:.8g} ± {stderrs[-1]:.3g}")


plt.errorbar(ps, [1 - m for m in means], yerr=stderrs, fmt='.', color='black', ecolor='black', capsize=3, elinewidth=1)
plt.xlabel("p")
plt.ylabel("Adj. Inflation Score $1-s$")
plt.show()

Simulating p = 0.10
Realisations: 24/25
Inflation Score: 1 ± 1.18e-09
Simulating p = 0.15
Failure:interrupted


SolverError: Solver 'SCS' failed. Try another solver, or solve with verbose=True for more information.

In [ ]:
from mipt import *
import warnings
import numpy as np
import importlib
import csv
import gnme
importlib.reload(gnme)
from gnme import RingInflationSDP3Q
from read_mipt_rho3 import read_rho3_bin

warnings.filterwarnings('ignore', category=UserWarning)

sdp: RingInflationSDP3Q

ps = np.linspace(0.0, 0.75, 16)
limit = 250
n = 8

print("Importing records... ", end='')
records = read_rho3_bin("rho3.bin", p_vals=ps, limit=limit)
print("done.", flush=True)

means = []
stderrs = []

sdp = RingInflationSDP3Q(domain="auto")
with open("gnme_lvl2.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["p", "score"])

for i in range(len(ps)):
    p = ps[i]
    scores = []
    print(f"Getting score for p = {p}...", flush=True)
    for r in range(limit):
        rhos = [records[i*limit + r][j]["rho"] for j in range(n)]
        gmn_scores = [gmn(rho, parties=3) for rho in rhos]
        idx = max(range(len(gmn_scores)), key=gmn_scores.__getitem__)
        rho = rhos[idx]

        score = sdp.solve(
            rho,
            gpu=False,
            verbose=False,
            max_iters=1500,
            tol=1e-8,
            time_limit_secs=45,
        )
        scores.append(score)
        new_row = [p, score]
        with open("gnme_lvl2.csv", mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(new_row)
        
        print(f"Realisations: {r}/{limit}", end='\r', flush=True)
        
    means.append(np.mean(scores))
    stderrs.append(np.std(scores)/np.sqrt(limit))
    print(f"\nInflation Score: {means[-1]:.8g} ± {stderrs[-1]:.3g}")
